# Registro y uso del modelo seleccionado
Ejecutar después de 02 y 03. No reentrena: carga el ganador por CV de una sesión completa. La muestra pertenece al test reservado y no constituye validación externa. Uso educativo.

In [0]:
import json
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
usuario = spark.sql('SELECT current_user() AS usuario').first()['usuario']
exp = mlflow.get_experiment_by_name(f'/Users/{usuario}/TP_Final_Insuficiencia_Cardiaca')
if exp is None:
    raise ValueError('Ejecutá primero 02.')
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id],
    filter_string="attributes.status = 'FINISHED' AND tags.study = 'heart_failure_extended_v1'",
    order_by=['attributes.start_time DESC'], max_results=1000)
counts = runs.groupby('tags.session_id')['tags.mlflow.runName'].nunique()
runs = runs[runs['tags.session_id'].isin(counts[counts == 8].index)]
if runs.empty:
    raise ValueError('Falta una sesión completa de ocho configuraciones.')
session_id = runs.iloc[0]['tags.session_id']
runs = runs[runs['tags.session_id'] == session_id].drop_duplicates('tags.mlflow.runName')
best = runs.sort_values('metrics.cv_roc_auc_mean', ascending=False).iloc[0]
run_id = best['run_id']
model_uri = f'runs:/{run_id}/modelo'
print('Ganador:', best['tags.mlflow.runName'], '| CV AUC:', best['metrics.cv_roc_auc_mean'])

Ganador: LR_barrido_C_10.0 | CV AUC: 0.8044444444444444


In [0]:
# Opcional: escribir catalogo.esquema.nombre en el widget para registrar en Unity Catalog.
# Vacío permite demostrar el uso del modelo directamente desde MLflow Tracking.
dbutils.widgets.text('registered_model_name', '', 'Nombre completo para registro (opcional)')
name = dbutils.widgets.get('registered_model_name').strip()
if name:
    mlflow.set_registry_uri('databricks-uc')
    version = mlflow.register_model(model_uri, name, await_registration_for=120)
    MlflowClient().set_registered_model_alias(name, 'champion', version.version)
    print('Registrado:', name, 'versión', version.version, 'alias champion')
    model_uri = f'models:/{name}@champion'
else:
    print('Carga desde el run; registro opcional sin ejecutar.')
model = mlflow.sklearn.load_model(model_uri)

Carga desde el run; registro opcional sin ejecutar.


In [0]:
partition_path = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path='particion.json')
with open(partition_path) as f:
    partition = json.load(f)
df = pd.read_csv('/Volumes/workspace/bigdata_drrach/landing/tp_final_heart_failure/heart_failure.csv')
features = partition['features']
sample = df.loc[partition['test_indices'], features].copy()
threshold = float(best['params.threshold'])
scores = model.predict_proba(sample)[:, 1]
predictions = pd.DataFrame({'fila_dataset': sample.index,
    'score_DEATH_EVENT': scores, 'prediccion': (scores >= threshold).astype(int),
    'umbral': threshold, 'run_id': run_id, 'model_uri': model_uri})
display(predictions.head(10))
# Verificar que el modelo recargado reproduce las predicciones registradas.
original_path = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path='evaluacion/predicciones_test.csv')
original = pd.read_csv(original_path)
assert np.allclose(scores, original['probabilidad']), 'Las predicciones no coinciden; revisar datos y entorno.'
base = '/Volumes/workspace/bigdata_drrach/landing/tp_final_heart_failure/resultados'
dbutils.fs.mkdirs(base)
predictions.to_csv(base + '/predicciones_modelo_recargado.csv', index=False)
print('Predicciones reproducidas y exportadas:', len(predictions))

fila_dataset,score_DEATH_EVENT,prediccion,umbral,run_id,model_uri
274,0.6409082447437399,1,0.5,1a3585d808fb46ab987ef1dbc564c5df,runs:/1a3585d808fb46ab987ef1dbc564c5df/modelo
149,0.42233561851190116,0,0.5,1a3585d808fb46ab987ef1dbc564c5df,runs:/1a3585d808fb46ab987ef1dbc564c5df/modelo
120,0.3471621341380737,0,0.5,1a3585d808fb46ab987ef1dbc564c5df,runs:/1a3585d808fb46ab987ef1dbc564c5df/modelo
226,0.6516090560787613,1,0.5,1a3585d808fb46ab987ef1dbc564c5df,runs:/1a3585d808fb46ab987ef1dbc564c5df/modelo
125,0.12299698550274768,0,0.5,1a3585d808fb46ab987ef1dbc564c5df,runs:/1a3585d808fb46ab987ef1dbc564c5df/modelo
298,0.13960828296990913,0,0.5,1a3585d808fb46ab987ef1dbc564c5df,runs:/1a3585d808fb46ab987ef1dbc564c5df/modelo
130,0.1782272579366935,0,0.5,1a3585d808fb46ab987ef1dbc564c5df,runs:/1a3585d808fb46ab987ef1dbc564c5df/modelo
110,0.18708523462867882,0,0.5,1a3585d808fb46ab987ef1dbc564c5df,runs:/1a3585d808fb46ab987ef1dbc564c5df/modelo
275,0.13259258386411077,0,0.5,1a3585d808fb46ab987ef1dbc564c5df,runs:/1a3585d808fb46ab987ef1dbc564c5df/modelo
190,0.7246269259847266,1,0.5,1a3585d808fb46ab987ef1dbc564c5df,runs:/1a3585d808fb46ab987ef1dbc564c5df/modelo


Predicciones reproducidas y exportadas: 60


## Alcance
El score corresponde a DEATH_EVENT durante seguimientos de diferente duración, no a mortalidad a un plazo fijo. No se dispone de una cohorte nueva: no se informa monitoreo real ni validación externa. El umbral 0.5 no está optimizado clínicamente. Para monitoreo futuro se requerirían nuevos pacientes y sus desenlaces.